# Model Context Protocol (MCP) Concepts

The **Model Context Protocol (MCP)** is an open protocol that standardizes how applications provide context to Large Language Models (LLMs). Think of it as a universal adapter that allows AI assistants to connect to various data sources and tools.

> 📝 **Hands-on Exercises**: After reviewing these concepts, complete the exercises in [EXERCISES.md](EXERCISES.md).

## Key Benefits
- **Standardization**: One protocol to connect to many tools
- **Security**: Controlled access to resources
- **Flexibility**: Works with any LLM provider

## MCP Architecture

```
+-----------------+     +-----------------+     +-----------------+
|   AI Agent      |---->|   MCP Client    |---->|   MCP Server    |
|   (Host)        |     |                 |     |   (Tools)       |
+-----------------+     +-----------------+     +-----------------+
```

### Components
1. **Host**: The AI application (e.g., Claude, ChatGPT integration, or custom agent)
2. **Client**: Connects to MCP servers on behalf of the host
3. **Server**: Exposes tools, resources, and prompts

## STDIO Transport (Local MCP)

In this lab, we focus on **STDIO (Standard Input/Output)** transport for local MCP servers.

### How STDIO Transport Works
- The MCP server runs as a **subprocess** spawned by the client
- Communication happens via **stdin/stdout** pipes
- Messages are exchanged using **JSON-RPC** format
- Best for: Local tools, CLI applications, development scenarios

### Architecture
```
AI Agent Client
    │
    │ spawns subprocess
    ▼
Local MCP Server (mcp_local_server)
    │
    │ stdin/stdout
    ▼
JSON-RPC Messages
    - initialize
    - tools/list
    - tools/call
```

### Advantages of STDIO
- **Simple setup**: No ports, no network configuration
- **Secure**: Communication stays within the local machine
- **Fast**: Direct process-to-process communication
- **Isolated**: Each client gets its own server instance

## Workshop Project Overview

| Project | Type | Transport | Description |
|---------|------|-----------|-------------|
| mcp_local_server | Local MCP Server | STDIO | Python MCP server with Ticket tools |
| mcp_remote_server | REST API Backend | HTTP | FastAPI REST API serving ticket data |
| mcp_bridge | Remote MCP Server | Streamable HTTP | MCP Bridge wrapping REST API |
| mcp_agent_client | AI Agent Client | STDIO + Streamable HTTP | AI agent that consumes both local and remote MCP servers |

## Setup: Install Required Packages

Run the following cell to install the required packages for MCP and Azure OpenAI integration.

In [ ]:
# Setup: Install required packages
# Run this cell first to ensure all dependencies are installed

import subprocess
import sys

packages = [
    "mcp",
    "openai",
    "azure-identity",
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    
print("✅ All packages installed successfully!")

## Configuration: Load Environment Variables

Set up your Azure OpenAI connection by loading from the `.env` file.

In [ ]:
# Configuration: Load environment variables
import os
import json
from pathlib import Path


def find_config_path(start_path: str) -> str:
    """Find the 'python' folder by traversing up from start_path."""
    current_dir = Path(start_path)
    
    while current_dir is not None:
        if current_dir.name.lower() == "python":
            return str(current_dir)
        if current_dir.parent == current_dir:
            break
        current_dir = current_dir.parent
    
    return start_path


def load_env_file(env_path: str) -> dict:
    """Load environment variables from .env file (JSON format)."""
    env_file = Path(env_path) / ".env"
    
    if not env_file.exists():
        return {}
    
    try:
        with open(env_file, 'r') as f:
            content = f.read()
            env_vars = json.loads(content)
            
            for key, value in env_vars.items():
                os.environ[key] = str(value)
            
            return env_vars
    except (json.JSONDecodeError, IOError) as e:
        print(f"Warning: Failed to load .env file: {e}")
        return {}


# Load environment variables
config_path = find_config_path(os.getcwd())
env_vars = load_env_file(config_path)
if env_vars:
    print(f"✅ Loaded {len(env_vars)} environment variables from: {config_path}/.env")
else:
    print("⚠️ No .env file found. Create labs/python/.env with your Azure OpenAI credentials.")

# Check required configuration
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT") or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")

if endpoint:
    print(f"✅ Azure OpenAI Endpoint: {endpoint}")
    print(f"✅ Deployment Name: {deployment}")
else:
    print("⚠️ Azure endpoint not set")

api_key = os.getenv("AZURE_OPENAI_API_KEY")
if api_key:
    print("✅ Authentication: API Key")
else:
    print("✅ Authentication: Azure CLI / DefaultAzureCredential")

## Create Azure OpenAI Client

This demonstrates how to set up the Azure OpenAI client.

In [ ]:
# Create Azure OpenAI Client
import os
from openai import AzureOpenAI
from azure.identity import AzureCliCredential

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o-mini")

if not endpoint:
    raise ValueError("Azure endpoint not set. Set AZURE_OPENAI_ENDPOINT environment variable.")

api_version = "2024-02-15-preview"

if api_key:
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version
    )
    print(f"✅ Created AzureOpenAI client with API Key auth")
else:
    credential = AzureCliCredential()
    token = credential.get_token("https://cognitiveservices.azure.com/.default")
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=token.token,
        api_version=api_version
    )
    print("✅ Created AzureOpenAI client with Azure CLI auth")

print(f"   Endpoint: {endpoint}")
print(f"   Deployment: {deployment}")

## Test the Azure OpenAI Connection

Send a simple completion request to verify everything is working.

In [ ]:
# Test the Azure OpenAI connection
response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say 'Hello from MCP Lab!' in exactly 5 words."}
    ],
    max_tokens=50
)

print("Response from Azure OpenAI:")
print(f"  {response.choices[0].message.content}")
print()
print(f"✅ Azure OpenAI connection successful!")
print(f"  Model: {response.model}")
print(f"  Tokens: {response.usage.total_tokens}")

## MCP Server Example - Define Tools

This shows how to create MCP tools using the `@server.list_tools()` decorator. In the actual exercise, you'll define these in `mcp_local_server/main.py`.

In [ ]:
# MCP Server Example - Define tools
from mcp.server import Server
from mcp.types import Tool, TextContent

# Create an MCP server instance
server = Server("example-server")

# Define tools using the list_tools decorator
@server.list_tools()
async def list_tools() -> list[Tool]:
    """List available tools."""
    return [
        Tool(
            name="GetAllTickets",
            description="Gets all support tickets with optional limit",
            inputSchema={
                "type": "object",
                "properties": {
                    "maxResults": {
                        "type": "integer",
                        "description": "Maximum number of tickets to return (default: 5)",
                        "default": 5
                    }
                },
                "required": []
            }
        ),
        Tool(
            name="GetTicket",
            description="Gets a support ticket by ID",
            inputSchema={
                "type": "object",
                "properties": {
                    "ticket_id": {
                        "type": "string",
                        "description": "The ticket ID (e.g., TICKET-001)"
                    }
                },
                "required": ["ticket_id"]
            }
        ),
        Tool(
            name="UpdateTicket",
            description="Updates a support ticket status",
            inputSchema={
                "type": "object",
                "properties": {
                    "ticket_id": {
                        "type": "string",
                        "description": "The ticket ID"
                    },
                    "status": {
                        "type": "string",
                        "description": "The new status (Open, In Progress, Resolved, Closed)"
                    }
                },
                "required": ["ticket_id", "status"]
            }
        )
    ]

print("✅ MCP Server defined with tools")
print("Available tools:")
tools = await list_tools()
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")

## MCP Tool Handler Example

This demonstrates how the `@server.call_tool()` handler works. When an AI agent calls a tool, this handler executes the logic.

In [ ]:
# MCP Tool Handler Example
import json

# Sample ticket data (same as what's loaded from assets/tickets.json)
sample_tickets = {
    "TICKET-001": {"id": "TICKET-001", "subject": "Login issue", "status": "Open", "description": "Cannot login"},
    "TICKET-002": {"id": "TICKET-002", "subject": "Performance", "status": "In Progress", "description": "System slow"},
}

# Simulating what @server.call_tool() does
async def handle_tool_call(name: str, arguments: dict) -> str:
    """Handle tool calls - this is what the call_tool handler does."""
    
    if name == "GetAllTickets":
        max_results = arguments.get("maxResults", 5)
        all_tickets = list(sample_tickets.values())[:max_results]
        return json.dumps(all_tickets, indent=2)
    
    elif name == "GetTicket":
        ticket_id = arguments.get("ticket_id", "")
        if ticket_id in sample_tickets:
            return json.dumps(sample_tickets[ticket_id], indent=2)
        return f"Ticket '{ticket_id}' not found"
    
    elif name == "UpdateTicket":
        ticket_id = arguments.get("ticket_id", "")
        status = arguments.get("status", "")
        if ticket_id in sample_tickets:
            sample_tickets[ticket_id]["status"] = status
            return f"Ticket '{ticket_id}' status updated to '{status}'"
        return f"Ticket '{ticket_id}' not found"
    
    return f"Unknown tool: {name}"

# Test the tool handler
print("=== Testing GetTicket ===")
result = await handle_tool_call("GetTicket", {"ticket_id": "TICKET-001"})
print(result)

print("\n=== Testing UpdateTicket ===")
result = await handle_tool_call("UpdateTicket", {"ticket_id": "TICKET-001", "status": "Resolved"})
print(result)

print("\n=== Testing GetAllTickets ===")
result = await handle_tool_call("GetAllTickets", {})
print(result)

## STDIO Transport: How It Works

The STDIO transport is the key mechanism for local MCP communication.

### Connection Flow

```
1. Client spawns server as subprocess:
   python -m mcp_local_server.main

2. Client sends JSON-RPC via stdin:
   {"jsonrpc": "2.0", "method": "initialize", ...}

3. Server responds via stdout:
   {"jsonrpc": "2.0", "result": {"capabilities": {...}}, ...}

4. Client discovers tools:
   {"jsonrpc": "2.0", "method": "tools/list", ...}

5. Client calls tools:
   {"jsonrpc": "2.0", "method": "tools/call", 
    "params": {"name": "GetTicket", "arguments": {"ticket_id": "TICKET-001"}}}
```

### Python Client Code Pattern

```python
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Define how to start the server
server_params = StdioServerParameters(
    command=sys.executable,
    args=["-m", "mcp_local_server.main"],
    cwd="path/to/begin"
)

# Connect and use
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        result = await session.call_tool("GetTicket", {"ticket_id": "TICKET-001"})
```

## Streamable HTTP Transport (Remote MCP)

While STDIO is great for local servers, **Streamable HTTP** transport enables **remote** MCP servers accessible over the network.

### How Streamable HTTP Transport Works
- The MCP server runs as a standalone **HTTP service** (e.g., on port 5070)
- Communication happens via **HTTP POST** requests to a `/mcp` endpoint
- Responses can be streamed using **Server-Sent Events (SSE)** for long-running operations
- Best for: Remote services, microservice architectures, shared servers

### Architecture: MCP Bridge Pattern
```
AI Agent Client
    │
    │ HTTP POST to /mcp
    ▼
MCP Bridge Server (:5070)
    │
    │ httpx (HTTP client)
    ▼
REST API Backend (:5060)
    │
    ▼
Ticket Data (JSON)
```

### Why the Bridge Pattern?
The **MCP Bridge** wraps an existing REST API without modifying the backend:
- **No changes** to the REST API — it stays a normal HTTP API
- **MCP compliance** — the bridge translates MCP tool calls into REST API requests
- **Reusability** — any MCP-compatible client can use the bridge
- **Separation of concerns** — backend team owns the API, AI team owns the bridge

### Advantages of Streamable HTTP
- **Network accessible**: Serve tools to remote clients
- **Scalable**: Can be deployed behind load balancers
- **Streaming**: SSE support for real-time updates
- **Standard HTTP**: Works with existing infrastructure (proxies, firewalls, auth)

## Remote MCP Client Code Pattern

### Connecting via Streamable HTTP

```python
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Connect to the remote MCP Bridge
url = "http://localhost:5070/mcp"

async with streamablehttp_client(url) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        
        # Discover tools (same API as STDIO!)
        tools = await session.list_tools()
        
        # Call a tool (same API as STDIO!)
        result = await session.call_tool("GetTicket", {"ticket_id": "TICKET-001"})
```

### Key Insight: Same Client API

Notice that the **client code is nearly identical** for both transports:

| Aspect | STDIO (Local) | Streamable HTTP (Remote) |
|--------|---------------|--------------------------|
| Import | `from mcp.client.stdio import stdio_client` | `from mcp.client.streamable_http import streamablehttp_client` |
| Connect | `stdio_client(server_params)` | `streamablehttp_client(url)` |
| Session API | `session.list_tools()` | `session.list_tools()` |
| Tool calls | `session.call_tool(...)` | `session.call_tool(...)` |

The **session API is identical** — this is the power of MCP's transport abstraction!

## AI Agent + MCP Integration Pattern

The agent client demonstrates how Azure OpenAI integrates with MCP tools. The pattern is **identical** whether using STDIO or Streamable HTTP:

```
User Query: "What's the status of TICKET-001?"
    │
    ▼
Azure OpenAI (with tool definitions)
    │
    ▼  decides to call GetTicket
MCP Client → [STDIO or HTTP] → MCP Server
    │                              │
    │                         executes tool
    │                              │
    ◀────── tool result ──────────┘
    │
    ▼
Azure OpenAI (formats response)
    │
    ▼
Agent: "TICKET-001 is currently Open."
```

### Key Steps in Code
1. **Discover tools** from MCP server via `session.list_tools()`
2. **Convert to OpenAI format** - map MCP tool schemas to OpenAI function definitions
3. **Send to Azure OpenAI** with tools and user message
4. **Handle tool calls** - when the AI decides to use a tool, call it via MCP
5. **Return results** - send tool output back to Azure OpenAI for final response

### Transport is Transparent
The agent client code for steps 1-5 is **exactly the same** regardless of transport. Only the **connection setup** differs (STDIO subprocess vs HTTP URL).

## Closing Thoughts

### What You've Learned

1. **MCP Protocol**: Standardized way for AI agents to connect to tools
2. **STDIO Transport**: Local, fast, secure subprocess communication
3. **Streamable HTTP Transport**: Remote, scalable, network-accessible communication
4. **MCP Bridge Pattern**: Wrapping existing REST APIs as MCP servers without modifying them
5. **Tool Definitions**: Schema-based tool descriptions for LLM understanding
6. **Tool Handlers**: Server-side execution of tool calls
7. **AI Integration**: Connecting MCP tools to Azure OpenAI for intelligent tool selection
8. **Transport Abstraction**: Same client API works for both local and remote servers

### Architecture Patterns

```
Local MCP (Exercises 1-3):
AI Agent Client → STDIO → Local MCP Server → Ticket Data

Remote MCP (Exercises 4-6):
AI Agent Client → Streamable HTTP → MCP Bridge (:5070) → REST API (:5060) → Ticket Data
```

### Key Takeaway

**MCP is a universal adapter layer** that lets AI agents interact with any tool or data source through a standardized protocol. Whether tools are **local** (STDIO) or **remote** (Streamable HTTP), the client-side API stays the same — enabling flexible, scalable AI agent architectures.

### Next Steps

1. Complete the hands-on exercises in [EXERCISES.md](EXERCISES.md)
2. Add more tools to the local and remote servers
3. Review the MCP specification at [modelcontextprotocol.io](https://modelcontextprotocol.io)